In [ ]:
%matplotlib qt

Point 1: x=96.8, y=53.5
Point 2: x=94.7, y=98.4
Point 3: x=98.9, y=136.3
Point 4: x=104.5, y=162.2
Point 5: x=123.5, y=194.5
Point 6: x=140.3, y=215.5
Point 1: x=110.8, y=54.9
Point 2: x=101.7, y=97.7
Point 3: x=106.6, y=136.3
Point 4: x=108.7, y=160.1
Point 5: x=126.3, y=191.7
Point 1: x=115.7, y=53.5
Point 2: x=106.6, y=102.6
Point 3: x=108.0, y=134.9
Point 4: x=122.8, y=169.2
Point 1: x=98.2, y=57.0
Point 2: x=95.4, y=73.8
Point 3: x=93.3, y=88.6
Point 4: x=91.2, y=111.0
Point 5: x=92.6, y=130.6
Point 6: x=95.4, y=140.5
Point 7: x=103.1, y=151.7
Point 8: x=162.0, y=248.5
Point 1: x=93.5, y=45.6
Point 2: x=96.1, y=101.4
Point 3: x=79.1, y=105.5
Point 4: x=79.1, y=105.5
Point 5: x=83.7, y=143.7
Point 6: x=90.9, y=163.3
Point 7: x=105.9, y=183.0
Point 1: x=96.1, y=47.9
Point 2: x=93.3, y=101.2
Point 3: x=101.0, y=146.1
Point 1: x=106.6, y=53.5
Point 2: x=99.6, y=101.2
Point 1: x=95.1, y=52.4
Point 2: x=95.6, y=118.9
  → Cut 1 complete
Point 3: x=68.2, y=126.2
Point 4: x=80.6, y=169.5
 

In [ ]:
import sunpy.map
import matplotlib.pyplot as plt
import glob
import pandas as pd
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from scipy.signal import find_peaks

#1. find the closest -short file to a given regular file

def find_closest_short(
    regular_map,
    short_dir="../data/20260820T154500"
):

    short_files = sorted(
        glob.glob(
            f"{short_dir}/solo_L2_eui-hrieuv174-image-short_*.fits"
        )
    )

    target = pd.Timestamp(regular_map.date.iso)

    short_times = []

    for f in short_files:

        filename = f.split("/")[-1]

        timestamp = filename.split("image-short_")[1].split("_V")[0]

        short_times.append(
            pd.to_datetime(
                timestamp,
                format="%Y%m%dT%H%M%S%f"
            )
        )

    closest_idx = np.argmin(
        np.abs(
            np.array(short_times, dtype="datetime64[ns]")
            - target.to_datetime64()
        )
    )

    closest_file = short_files[closest_idx]

    print("Closest short file:", closest_file)
    print("Time difference:", abs(short_times[closest_idx] - target))

    return sunpy.map.Map(closest_file)


#2. make the 150 x 150 arcsec zoom

def make_zoom(map_obj):

    zoom_width = 150 * u.arcsec
    zoom_height = 150 * u.arcsec

    bottom_left = SkyCoord(
        -1370 * u.arcsec - zoom_width / 2,
        375 * u.arcsec - zoom_height / 2,
        frame=map_obj.coordinate_frame
    )

    top_right = SkyCoord(
        -1370 * u.arcsec + zoom_width / 2,
        375 * u.arcsec + zoom_height / 2,
        frame=map_obj.coordinate_frame
    )

    return map_obj.submap(
        bottom_left,
        top_right=top_right
    )

#3. pick slice points on the three zoom sections

from matplotlib.colors import LogNorm

def pick_points(map_obj):

    #define the three zoom regions
    
    zoom_regions = [
        ("Zoom A — lower", -1400, 330),
        ("Zoom B — middle", -1390, 375),
        ("Zoom C — upper", -1360, 430),
    ]

    zoom_size = 80 * u.arcsec

    
    fig = plt.figure(figsize=(15, 5))

    axes = []
    clicked = []

    #make the three interactive panels
    for i, (title, tx, ty) in enumerate(zoom_regions):

        bottom_left = SkyCoord(
            tx * u.arcsec - zoom_size / 2,
            ty * u.arcsec - zoom_size / 2,
            frame=map_obj.coordinate_frame
        )

        top_right = SkyCoord(
            tx * u.arcsec + zoom_size / 2,
            ty * u.arcsec + zoom_size / 2,
            frame=map_obj.coordinate_frame
        )

        zoom_map = map_obj.submap(
            bottom_left,
            top_right=top_right
        )

        ax = fig.add_subplot(
            1, 3, i + 1,
            projection=zoom_map.wcs
        )

        axes.append(ax)

        #logarithmic intensity stretch
    
        data = zoom_map.data

        vmin = np.nanpercentile(
            data,
            35
        )

        vmax = np.nanpercentile(
            data,
            99.5
        )

        zoom_map.plot(
            axes=ax,
            norm=LogNorm(
                vmin=vmin,
                vmax=vmax
            )
        )

        ax.set_title(title)

    def onclick(event):

        if event.inaxes not in axes:
            return

        ax = event.inaxes

        clicked.append(
            (
                event.inaxes,
                event.xdata,
                event.ydata
            )
        )

        ax.plot(
            event.xdata,
            event.ydata,
            "ro",
            markersize=5
        )

        #draw line for each pair
        if len(clicked) % 2 == 0:

            ax1, x1, y1 = clicked[-2]
            ax2, x2, y2 = clicked[-1]

            #only connect points if they were clicked in the same zoom panel
            if ax1 == ax2:

                ax.plot(
                    [x1, x2],
                    [y1, y2],
                    "r-",
                    linewidth=2
                )

                print(
                    f"  → Cut {len(clicked)//2} complete"
                )

            else:

                print(
                    "WARNING: points for a cut must "
                    "be in the same zoom panel."
                )

        fig.canvas.draw_idle()

        print(
            f"Point {len(clicked)}: "
            f"x={event.xdata:.1f}, "
            f"y={event.ydata:.1f}"
        )

    fig.canvas.mpl_connect(
        "button_press_event",
        onclick
    )

    plt.tight_layout()

    return axes, clicked, zoom_regions


# 4. extract intensity profile along the path

def extract_profile(map_obj, points):

    #convert the SkyCoord path into the map pixels intersected by it
    intensity_coords = sunpy.map.pixelate_coord_path(
        map_obj,
        points
    )

    #extract intensity values at those pixel coordinates
    intensity = sunpy.map.sample_at_coords(
        map_obj,
        intensity_coords
    ).value

    #angular distance from the start of the slice
    distance = (
        intensity_coords.separation(
            intensity_coords[0]
        ).to(u.arcsec)
    )

    return intensity_coords, intensity, distance


In [ ]:
from scipy.signal import find_peaks
import numpy as np


km_per_pixel = 106
pixel_scale = 0.492


def find_peak_widths(distance, intensity):

    x = distance.value
    y = intensity

    peaks, properties = find_peaks(
        y,
        prominence=np.std(y) * 0.005,
        distance=2,
        width=1
    )

    measurements = []

    
    #extract SciPy width measurements

    for i, peak in enumerate(peaks):

        width_pixels = properties["widths"][i]

        width_arcsec = (
            width_pixels *
            pixel_scale
        )

        width_km = (
            width_pixels *
            km_per_pixel
        )

        measurements.append({
            "peak_index": peak,
            "peak_position_arcsec": x[peak],
            "prominence": properties["prominences"][i],
            "width_pixels": width_pixels,
            "width_arcsec": width_arcsec,
            "width_km": width_km,
            "left_ips": properties["left_ips"][i],
            "right_ips": properties["right_ips"][i],
            "width_height": properties["width_heights"][i],
        })

    return measurements

In [202]:


regular_map = sunpy.map.Map(
    "../data/20260820T154500/"
    "solo_L2_eui-hrieuv174-image_20260820T162956285_V00.fits"
)

short_map = find_closest_short(regular_map)

axes, clicked, zoom_regions = pick_points(regular_map)

print(f"Picked {len(clicked)} points")

Closest short file: ../data/20260820T154500\solo_L2_eui-hrieuv174-image-short_20260820T162954285_V00.fits
Time difference: 0 days 00:00:02
Picked 0 points


In [ ]:
# ============================================================
# Create transverse cuts from IMAGE clicks
# ============================================================

cuts = []

for i in range(0, len(clicked), 2):

    ax1, x1, y1 = clicked[i]
    ax2, x2, y2 = clicked[i + 1]

    #make sure both points belong to the same panel
    if ax1 != ax2:

        print(
            f"Skipping cut {i//2 + 1}: "
            "points are in different panels."
        )

        continue

    # Convert the clicked pixel coordinates
    # back into SkyCoord using the correct zoom map
    # for this panel

    panel_index = axes.index(ax1)

    tx, ty = zoom_regions[panel_index][1:]

    zoom_size = 80 * u.arcsec

    bottom_left = SkyCoord(
        tx * u.arcsec - zoom_size / 2,
        ty * u.arcsec - zoom_size / 2,
        frame=regular_map.coordinate_frame
    )

    top_right = SkyCoord(
        tx * u.arcsec + zoom_size / 2,
        ty * u.arcsec + zoom_size / 2,
        frame=regular_map.coordinate_frame
    )

    panel_map = regular_map.submap(
        bottom_left,
        top_right=top_right
    )

    x = np.array([x1, x2]) * u.pix
    y = np.array([y1, y2]) * u.pix

    cut = panel_map.pixel_to_world(
        x,
        y
    )

    cuts.append(cut)

    print(
        f"Cut {len(cuts)} created from "
        f"{zoom_regions[panel_index][0]}"
    )

Cut 1 created from Zoom A — lower
Cut 2 created from Zoom A — lower
Cut 3 created from Zoom A — lower
Cut 4 created from Zoom A — lower
Cut 5 created from Zoom A — lower
Cut 6 created from Zoom B — middle
Cut 7 created from Zoom B — middle
Cut 8 created from Zoom B — middle
Cut 9 created from Zoom B — middle
Cut 10 created from Zoom B — middle
Cut 11 created from Zoom C — upper
Cut 12 created from Zoom C — upper
Cut 13 created from Zoom C — upper


In [235]:
# ============================================================
# PLOT INTENSITY PROFILES + CLASSIFIED PEAKS FOR ALL CUTS
# ============================================================

fig, axes_profiles = plt.subplots(
    len(cuts),
    1,
    figsize=(10, 3 * len(cuts))
)

# Make sure axes_profiles is always iterable
if len(cuts) == 1:
    axes_profiles = [axes_profiles]


# ------------------------------------------------------------
# Suspicious peaks
# ------------------------------------------------------------

suspicious = [
    (8, 4),   # 7.26 px
    (6, 4),   # 6.92 px
    (6, 1),   # 6.09 px
    (1, 3),   # 4.89 px
    (7, 4),   # 4.54 px
]


# ------------------------------------------------------------
# Loop through all cuts
# ------------------------------------------------------------

for i, cut in enumerate(cuts):

    cut_number = i + 1

    coords, intensity, distance = extract_profile(
        regular_map,
        cut
    )

    # IMPORTANT:
    # Keep exactly the same detection settings as before
    peaks, properties = find_peaks(
        intensity,
        prominence=np.std(intensity) * 0.005,
        distance=2
    )

    ax = axes_profiles[i]

    # --------------------------------------------------------
    # Plot intensity profile
    # --------------------------------------------------------

    ax.plot(
        distance.value,
        intensity,
        drawstyle="steps-mid",
        label="Intensity profile"
    )

    # --------------------------------------------------------
    # Work out accepted peak numbers
    # --------------------------------------------------------

    selection = accepted_peaks[cut_number]

    if selection == "all":

        accepted_numbers = list(
            range(1, len(peaks) + 1)
        )

    elif selection == "all_except_first":

        accepted_numbers = list(
            range(2, len(peaks) + 1)
        )

    elif selection == "all_except_first_and_last_two":

        accepted_numbers = list(
            range(2, len(peaks) - 1)
        )

    else:

        accepted_numbers = selection

    # --------------------------------------------------------
    # Classify every detected peak
    # --------------------------------------------------------

    accepted_x = []
    accepted_y = []

    suspicious_x = []
    suspicious_y = []

    discarded_x = []
    discarded_y = []

    for peak_index, peak_position in enumerate(peaks):

        # Peak numbering starts at 1
        peak_number = peak_index + 1

        x = distance[peak_position].value
        y = intensity[peak_position]

        if (cut_number, peak_number) in suspicious:

            suspicious_x.append(x)
            suspicious_y.append(y)

        elif peak_number in accepted_numbers:

            accepted_x.append(x)
            accepted_y.append(y)

        else:

            discarded_x.append(x)
            discarded_y.append(y)

    # --------------------------------------------------------
    # Accepted peaks
    # --------------------------------------------------------

    if accepted_x:

        ax.plot(
            accepted_x,
            accepted_y,
            "ro",
            markersize=5,
            label="Accepted peaks"
        )

    # --------------------------------------------------------
    # Suspicious peaks
    # --------------------------------------------------------

    if suspicious_x:

        ax.plot(
            suspicious_x,
            suspicious_y,
            "o",
            color="orange",
            markersize=6,
            label="Suspicious peaks"
        )

    # --------------------------------------------------------
    # Discarded peaks
    # --------------------------------------------------------

    if discarded_x:

        ax.plot(
            discarded_x,
            discarded_y,
            "o",
            color="black",
            markersize=4,
            label="Discarded peaks"
        )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel(
        "Distance along transverse cut [arcsec]"
    )

    ax.set_ylabel(
        "Intensity [DN/s]"
    )

    ax.set_title(
        f"Cut {cut_number} — {len(peaks)} peaks detected"
    )

    ax.legend()

    ax.grid(
        alpha=0.3
    )


plt.tight_layout()

plt.savefig(
    "../figures/logstretch_intensity_profiles.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# ACCEPTED PEAKS FOR EACH TRANSVERSE CUT
# ============================================================

accepted_peaks = {
    1: [3, 5, 6],
    2: [2, 3, 4, 5],
    3: "all",
    4: [2, 3],
    5: [2],
    6: "all",
    7: "all_except_first",
    8: [3, 4, 5],
    9: [2],
    10: [2, 4],
    11: "all",
    12: "all_except_first_and_last_two",
    13: [3, 4, 5, 6, 7]
}

In [ ]:
# ============================================================
# PLOT ZOOMS WITH ACCEPTED PEAKS + CUT LINES
# ============================================================

from matplotlib.colors import LogNorm

overview_width = 250 * u.arcsec
overview_height = 250 * u.arcsec

bottom_left = SkyCoord(
    -1370 * u.arcsec - overview_width / 2,
    375 * u.arcsec - overview_height / 2,
    frame=regular_map.coordinate_frame
)

top_right = SkyCoord(
    -1370 * u.arcsec + overview_width / 2,
    375 * u.arcsec + overview_height / 2,
    frame=regular_map.coordinate_frame
)

overview_map = regular_map.submap(
    bottom_left,
    top_right=top_right
)

vmin = np.nanpercentile(
    overview_map.data,
    35
)

vmax = np.nanpercentile(
    overview_map.data,
    99.5
)

fig = plt.figure(
    figsize=(15, 5)
)

zoom_size = 80 * u.arcsec

# ------------------------------------------------------------
# Store accepted peak coordinates AND cut coordinates
# ------------------------------------------------------------

peak_positions = {
    0: [],
    1: [],
    2: []
}

cut_positions = {
    0: [],
    1: [],
    2: []
}

# ------------------------------------------------------------
# Extract accepted peaks and corresponding cut lines
# ------------------------------------------------------------

for cut_number, cut in enumerate(cuts, start=1):

    coords, intensity, distance = extract_profile(
        regular_map,
        cut
    )

    peaks, properties = find_peaks(
        intensity,
        prominence=np.std(intensity) * 0.005,
        distance=2
    )

    selection = accepted_peaks[cut_number]

    if selection == "all":

        selected_peak_numbers = list(
            range(1, len(peaks) + 1)
        )

    elif selection == "all_except_first":

        selected_peak_numbers = list(
            range(2, len(peaks) + 1)
        )

    elif selection == "all_except_first_and_last_two":

        selected_peak_numbers = list(
            range(2, len(peaks) - 1)
        )

    else:

        selected_peak_numbers = selection

    # --------------------------------------------------------
    # Find which zoom panel this cut belongs to
    # --------------------------------------------------------

    panel_ax = clicked[
        (cut_number - 1) * 2
    ][0]

    panel_index = axes.index(panel_ax)

    # --------------------------------------------------------
    # Store the cut itself
    # --------------------------------------------------------

    cut_positions[panel_index].append(
        SkyCoord(cut)
    )

    # --------------------------------------------------------
    # Store accepted peak positions
    # --------------------------------------------------------

    for peak_number in selected_peak_numbers:

        peak_index = peak_number - 1

        if peak_index >= len(peaks):

            print(
                f"WARNING: Cut {cut_number} "
                f"requested peak {peak_number}, "
                f"but only {len(peaks)} peaks were detected."
            )

            continue

        peak_coord = coords[
            peaks[peak_index]
        ]

        peak_positions[
            panel_index
        ].append(peak_coord)


# ============================================================
# PLOT THE THREE ZOOM PANELS
# ============================================================

for i, (title, tx, ty) in enumerate(
    zoom_regions
):

    bottom_left = SkyCoord(
        tx * u.arcsec - zoom_size / 2,
        ty * u.arcsec - zoom_size / 2,
        frame=regular_map.coordinate_frame
    )

    top_right = SkyCoord(
        tx * u.arcsec + zoom_size / 2,
        ty * u.arcsec + zoom_size / 2,
        frame=regular_map.coordinate_frame
    )

    zoom_map = regular_map.submap(
        bottom_left,
        top_right=top_right
    )

    ax = fig.add_subplot(
        1,
        3,
        i + 1,
        projection=zoom_map.wcs
    )

    # --------------------------------------------------------
    # Plot image
    # --------------------------------------------------------

    zoom_map.plot(
        axes=ax,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax
        )
    )

    # --------------------------------------------------------
    # Plot ALL slice lines
    # --------------------------------------------------------

    for cut_coords in cut_positions[i]:

        x_cut, y_cut = zoom_map.world_to_pixel(
            cut_coords
        )

        ax.plot(
            x_cut,
            y_cut,
            color="red",
            linewidth=1.0,
            alpha=0.8
        )

    # --------------------------------------------------------
    # Plot accepted peak dots
    # --------------------------------------------------------

    if len(peak_positions[i]) > 0:

        peak_coords = SkyCoord(
            peak_positions[i]
        )

        x_peak, y_peak = zoom_map.world_to_pixel(
            peak_coords
        )

        ax.plot(
            x_peak,
            y_peak,
            "ro",
            markersize=5
        )

    ax.set_title(title)


plt.tight_layout()

plt.savefig(
    "../figures/logstretch_peaks.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [210]:
# ============================================================
# NEW RESULTS TABLE — SCiPY PEAK WIDTHS
# ============================================================

import pandas as pd

results = []

In [ ]:
accepted_peaks = {
    1: [3, 5, 6],
    2: [2, 3, 4, 5],
    3: "all",
    4: [2, 3],
    5: [2],
    6: "all",
    7: "all_except_first",
    8: [3, 4, 5],
    9: [2],
    10: [2, 4],
    11: "all",
    12: "all_except_first_and_last_two",
    13: [3, 4, 5, 6, 7]
}

In [ ]:
# ============================================================
# MEASURE WIDTHS OF ACCEPTED PEAKS
# ============================================================

from scipy.signal import find_peaks, peak_widths

km_per_pixel = 106
pixel_scale = 0.492

results = []

for cut_number, cut in enumerate(cuts, start=1):

    coords, intensity, distance = extract_profile(
        regular_map,
        cut
    )

    peaks, properties = find_peaks(
        intensity,
        prominence=np.std(intensity) * 0.005,
        distance=2
    )

   
    widths, width_heights, left_ips, right_ips = peak_widths(
        intensity,
        peaks,
        rel_height=0.5
    )

   
    selection = accepted_peaks[cut_number]

    if selection == "all":

        selected_peak_numbers = list(
            range(1, len(peaks) + 1)
        )

    elif selection == "all_except_first":

        selected_peak_numbers = list(
            range(2, len(peaks) + 1)
        )

    elif selection == "all_except_first_and_last_two":

        selected_peak_numbers = list(
            range(2, len(peaks) - 1)
        )

    else:

        selected_peak_numbers = selection

    
    for peak_number in selected_peak_numbers:

        peak_index = peak_number - 1

        # Safety check
        if peak_index >= len(peaks):

            print(
                f"WARNING: Cut {cut_number} "
                f"requested peak {peak_number}, "
                f"but only {len(peaks)} peaks were detected."
            )

            continue

        width_pixels = widths[peak_index]

        #convert to arcsec
        width_arcsec = (
            width_pixels * pixel_scale
        )

        #convert to km
        width_km = (
            width_pixels * km_per_pixel
        )

        results.append({
            "Cut": cut_number,
            "Peak": peak_number,
            "Prominence": properties[
                "prominences"
            ][peak_index],
            "Width [pixels]": width_pixels,
            "Width [arcsec]": width_arcsec,
            "Width [km]": width_km
        })

print(
    f"Total accepted peak measurements: {len(results)}"
)

Total accepted peak measurements: 46


In [215]:
# ============================================================
# CURRENT RESULTS TABLE
# ============================================================

results_table = pd.DataFrame(results)

results_table = results_table.round({
    "Prominence": 1,
    "Width [pixels]": 2,
    "Width [arcsec]": 2,
    "Width [km]": 1
})

results_table

,Cut,Peak,Prominence,Width [pixels],Width [arcsec],Width [km]
0,1,3,1145.2,4.89,2.41,518.3
1,1,5,770.7,1.71,0.84,181.5
2,1,6,360.1,3.50,1.72,370.8
3,2,2,8108.4,3.85,1.90,408.6
4,2,3,1481.2,1.02,0.50,108.5
5,2,4,807.4,1.07,0.53,113.8
6,2,5,256.7,0.60,0.30,63.9
7,3,1,1502.8,3.54,1.74,375.1
8,3,2,1774.5,2.72,1.34,287.9
9,3,3,1252.0,3.72,1.83,394.6


In [221]:
# ============================================================
# HISTOGRAM OF LOOP WIDTHS
# ============================================================

import matplotlib.pyplot as plt

widths_km = results_table["Width [km]"]

plt.figure(figsize=(8, 5))

plt.hist(
    widths_km,
    bins= 30,
    edgecolor="black"
)

plt.xlabel("Loop width [km]")
plt.ylabel("Number of measurements")
plt.title("Distribution of measured loop widths")

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [232]:
# ============================================================
# INSPECT SUSPICIOUSLY WIDE PEAKS
# ============================================================

from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from scipy.signal import find_peaks, peak_widths


suspicious = [
    (8, 4),   # 7.26 px
    (6, 4),   # 6.92 px
    (6, 1),   # 6.09 px
    (1, 3),   # 4.89 px
    (7, 4),   # 4.54 px
]


overview_width = 250 * u.arcsec
overview_height = 250 * u.arcsec

bottom_left = SkyCoord(
    -1370 * u.arcsec - overview_width / 2,
    375 * u.arcsec - overview_height / 2,
    frame=regular_map.coordinate_frame
)

top_right = SkyCoord(
    -1370 * u.arcsec + overview_width / 2,
    375 * u.arcsec + overview_height / 2,
    frame=regular_map.coordinate_frame
)

overview_map = regular_map.submap(
    bottom_left,
    top_right=top_right
)

vmin = np.nanpercentile(
    overview_map.data,
    35
)

vmax = np.nanpercentile(
    overview_map.data,
    99.5
)

fig = plt.figure(figsize=(15, 10))

for plot_number, (cut_number, peak_number) in enumerate(
    suspicious,
    start=1
):

    
    cut = cuts[cut_number - 1]

    coords, intensity, distance = extract_profile(
        regular_map,
        cut
    )

    
    peaks, properties = find_peaks(
        intensity,
        prominence=np.std(intensity) * 0.005,
        distance=2
    )

    widths, width_heights, left_ips, right_ips = peak_widths(
        intensity,
        peaks,
        rel_height=0.5
    )

    peak_index = peak_number - 1

   
    peak_coord = coords[
        peaks[peak_index]
    ]

    panel_ax = clicked[
        (cut_number - 1) * 2
    ][0]

    panel_index = axes.index(panel_ax)

    title, tx, ty = zoom_regions[panel_index]

   
    zoom_size = 80 * u.arcsec

    bottom_left = SkyCoord(
        tx * u.arcsec - zoom_size / 2,
        ty * u.arcsec - zoom_size / 2,
        frame=regular_map.coordinate_frame
    )

    top_right = SkyCoord(
        tx * u.arcsec + zoom_size / 2,
        ty * u.arcsec + zoom_size / 2,
        frame=regular_map.coordinate_frame
    )

    zoom_map = regular_map.submap(
        bottom_left,
        top_right=top_right
    )

  
    ax = fig.add_subplot(
        2,
        3,
        plot_number,
        projection=zoom_map.wcs
    )

    zoom_map.plot(
        axes=ax,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax
        )
    )

    x_peak, y_peak = zoom_map.world_to_pixel(
        peak_coord
    )

    ax.plot(
        x_peak,
        y_peak,
        "ro",
        markersize=4,
        markeredgecolor="white",
        markeredgewidth=1
    )

    
    cut_coords = SkyCoord(cut)

    x_cut, y_cut = zoom_map.world_to_pixel(
        cut_coords
    )

    ax.plot(
        x_cut,
        y_cut,
        "r-",
        linewidth=1.5
    )

   
    width = widths[peak_index]
    width_km = width * 106

    ax.set_title(
        f"Cut {cut_number}, Peak {peak_number}\n"
        f"Width = {width:.2f} px ({width_km:.0f} km)"
    )

plt.tight_layout()
plt.show()